In [1]:
import torch
from IPython.display import clear_output
from collections import deque

# 0. initialization:
words = open('names.txt', 'r').read().splitlines()
words[len(words) - 1000: len(words) - 990]
chars =sorted(list(set(''.join(words)))) # sorted list of all the characters

# creating i->s and s-> i:
s_to_i = {'.': 0}
for i in range(len(chars)):
    s_to_i[chars[i]] = i+1
i_to_s = {}
for char, idx in s_to_i.items():
    i_to_s[idx] = char

In [2]:
# 1. creating the dataset (filling in xs and ys):
xs, ys = [], []
for w in words:
    # preparing the extra char:
    chars = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chars, chars[1:]):
        idx1 = s_to_i[ch1]
        idx2 = s_to_i[ch2]
        xs.append(idx1)
        ys.append(idx2)

# convert xs and ys to tensors in the end:
xs = torch.tensor(xs)
ys = torch.tensor(ys)

print("Number of examples:", len(xs))

Number of examples: 228146


In [3]:
# 2. initialize the network:
g = torch.Generator().manual_seed(6767676767)
W = torch.randn((27,27), generator=g, requires_grad=True)

In [8]:
# 3. Backpropagation improvement loop:
epochs = 500
learning_rate = 70

recent_epochs = deque(maxlen=5) # for printing in the console

for epoch in range(epochs):
    # a. forward pass:
    xenc = torch.nn.functional.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)

    # b. calculating loss:
    likelihood = probs[torch.arange(len(xs)), ys] # proba of the right target appearing given an input
    nll = -likelihood.log()
    loss = nll.mean() # average NLL

    # c. resetting W's gradient to begin a wave of backprop
    W.grad = None

    # d. backpropagation simply
    loss.backward() # populates .grad of all the shizz

    # e. nudging knobs in the opposite dir of loss's gradient:
    W.data += W.grad * -learning_rate
    
    recent_epochs.append(f"Epoch {epoch + 1}/{epochs} - Loss: {loss:.4f}")
    clear_output(wait=True)
    print("\n".join(recent_epochs), flush=True)
        
print("current loss:", loss)

Epoch 496/500 - Loss: 2.4543
Epoch 497/500 - Loss: 2.4543
Epoch 498/500 - Loss: 2.4543
Epoch 499/500 - Loss: 2.4543
Epoch 500/500 - Loss: 2.4543
current loss: tensor(2.4543, grad_fn=<MeanBackward0>)


Benchmarking notes from above:
- 2k epochs, 50 learning: ~2.4543
- 5k epochs, 50 learning: same
- 5k epochs, 70 learning: jumped to 2.4542 within ~400 epochs, but literally stayed the same throughout - going to 2.4541 at the min
- 5k epochs, 80 learning: jumped to 2.4541 asap (kinda same as above), stayed same throughout

Now, we use this optimized W to actually come up with new names and shi

In [35]:
# Creating softmax from W:
W_exp = W.exp()
W_softmax = W_exp / W_exp.sum(1, keepdims=True)

new_name = ""
cur_index = 0

while True:
    # Picking next using softmax:
    # autoregression (current predicted output becomes next input) and sampling (multinomial)
    cur_index = torch.multinomial(W_softmax[cur_index], num_samples=1, replacement=True).item() 
    cur_letter = i_to_s[cur_index]
    new_name += cur_letter
    if cur_index == 0:
        break
new_name

'con.'

## Note 1: Why same loss across both approaches + why NN?
- the neural network approach achieves roughly the same loss (when NN is optimized) as the simpler method of normalizing counts to make them into probabilities
- this is because we are fundamentally dealing with the same information, + bigram language level model is super simple in the first place
- The usefullness of gradient-descent based approach comes in when we are taking not just one last character, but several characters to predict the next one (the structure of the full training loop for more complex will be ~the same, but with a little bit more complexity when doing things like the forward pass)
- The NN approach is also more scaleable compared to this bigram model (here, we know we store 2 chars proba in each spot. But what if we want to quickly see last 3, 4, ...? that's not scaleable in the non-NN approach)

## Note 2: Regularization
- it's a way to smooth the probability distribution
- think, in the original probabalistic model, instead of adding 1 to smooth to prevent NAs, we had some like 10000000 instead. This would make all of the probabilities in a row roughly the same - not what we want exactly to be particular but if done on a smaller scale can help us manage the scale and prevent super predictability stuff
- the way we apply smoothing to the Neural Net version is called "regularization". Deriving how regularization helps and where it fits:

## Fitting Regularization into our loss:
- if Ws are all 0s -> logits will be all 0s (matrix mult) -> counts will all be 1s (exponentiation) -> probability will be uniform (this is the extreme example of Ws all 0s)
- the idea is: we want Ws to not stray too far away from 0s that much to see a little bit more smoothed model:
    - achieved by optimizing the value of $\mu({W^2})$ - mean of all the W values and stuff, optimizing on this (like loss) will make more smoother proba distributions
    - We multiply this values by "regularization strength" (kind of like learning rate)

## Example Loss with Regularization included in:
```python
loss = nll.mean() + 0.01 * (W**2).mean() # average NLL + reg. strength * reg. val
```